# Week 5 Assignment Solutions

---
### Q1:  What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing? 

The biggest problem with MapReduce is that it relies heavily on disk. Every time it finishes a step, it writes the result to disk, and the next step reads it back. This makes it really slow.

Other issues:
- No support for real-time or interactive processing — everything is batch
- Very verbose code for even simple tasks
- Bad for iterative jobs (like ML) where you repeat the same steps many times

Spark fixes all this by keeping data in memory and providing simple, high-level APIs.

---
### Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

In iterative ML algorithms (e.g., Gradient Descent, k-Means), the same dataset is processed repeatedly across many iterations.

- **MapReduce:** Each iteration reads the dataset from disk, processes it, and writes results back to the disk. For 100 iterations, that's 100 disk reads and writes — extremely slow.
- **Spark:** The dataset is loaded into memory once using `.cache()` or `.persist()`. Each iteration reads directly from RAM, which is ~100x faster than disk.

---
### Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [ ]:
# Removing duplicates based on specific columns like user_id and transaction_date
df_cleaned = df.dropDuplicates(['user_id', 'transaction_date'])

# Removing exact duplicate rows (all columns)
df_cleaned = df.dropDuplicates()

df_cleaned.show()

`dropDuplicates()` keeps the first occurrence of each unique combination of the specified columns and drops the rest.

---
### Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [ ]:
from pyspark.sql.functions import col, avg

result = (
    df_sales
    .filter(col('region') == 'West')
    .groupBy('product_category')
    .agg(avg('sale_amount').alias('avg_sale_amount'))
)

result.show()

---
### Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

- **`.na.drop()`** – Removes rows that contain null values. You can specify a subset of columns or a threshold for how many nulls to tolerate.
- **`.na.fill()`** – Replaces null values with a specified default value instead of dropping the row. Useful when you want to retain the record but handle missing data.

In [ ]:
# Fill null values in 'status' column with 'Unknown'
df_filled = df.na.fill({'status': 'Unknown'})

df_filled.show()

In [ ]:
# Replace null values with 0 only in numeric columns (Integer/Float)
df_filled = df.na.fill(0)

# Replace null values with "Unknown" only in string columns
df_filled = df.na.fill("Unknown")

# Replace null values with False only in boolean columns
df_filled = df.na.fill(False)

In [ ]:
# keys represents column names, values represents replacement value
replacement_map = {
    "age": 0,
    "name": "Unknown",
    "city": "N/A",
    "is_active": True
}

df_filled = df.na.fill(replacement_map)

---
### Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [ ]:
from pyspark.sql.functions import col, count

result = (
    df
    .groupBy('city')
    .agg(count('*').alias('total_count'))
    .filter(col('total_count') > 100)
)

result.show()

The `.filter()` is applied after aggregation to only show cities exceeding the threshold — similar to `HAVING` in SQL.

---
### Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

Spark DataFrames are **immutable** that means we cannot modify them in place. Every transformation (like dropping a column or renaming) returns a **new** DataFrame.

This means for data cleaning:
- You must assign the result of each operation/transformation to a new variable (or overwrite the same one).
- It ensures data lineage is preserved — you can always trace back to the original.
- It also enables Spark's lazy evaluation.

In [ ]:
df_step1 = df.drop('unwanted_column')
df_step2 = df_step1.withColumnRenamed('old_name', 'new_name')

df_step2.show()

---
### Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [ ]:
from pyspark.sql.functions import col

result = df.filter(
    (col('age').between(18, 30)) & (col('subscription') == 'Premium')
)

result.show()

`.between(18, 30)` is inclusive on both ends, equivalent to `age >= 18 AND age <= 30`.

---
### Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

If null values are present in a column before aggregation:

- **`sum()`** will ignore nulls, which may give a misleading total.
- **`avg()`** will skip nulls when computing the average.
- **`count(column)`** counts only non-null values, which may differ from the actual row count.

Handling nulls beforehand (either by dropping rows or filling with a default value like 0 or mean) ensures that aggregation results are accurate and consistent.

---
### Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

df_updated = df.withColumn(
    'event_time',
    col('raw_timestamp').cast(TimestampType())
).drop('raw_timestamp')

df_updated.printSchema()
df_updated.show()

---
### Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

When a `groupBy()` is performed, Spark needs to bring all records with the **same key** onto the **same partition/executor**. This requires data to move across the network between worker nodes — this movement is called a **shuffle**.

**Why it's a wide transformation:**
- A *narrow transformation* (like `filter`, `map`) only works on data within a single partition — no data movement needed.
- A *wide transformation* (like `groupBy`, `join`, `distinct`) requires data from **multiple partitions** to be combined — triggering a shuffle.

Shuffles are expensive because they involve:
1. Serializing data
2. Sending it over the network
3. Writing shuffle files to disk temporarily
4. Deserializing and re-partitioning on the receiving end

---
### Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [ ]:
from pyspark.sql.functions import col

df_cleaned = df.filter(
    col('email').isNotNull() & (col('username') != '')
)

df_cleaned.show()

---
### Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [ ]:
from pyspark.sql.functions import min, max, mean

result = df.agg(
    min('price').alias('min_price'),
    max('price').alias('max_price'),
    mean('price').alias('mean_price')
)

result.show()

---
### Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

When `inferSchema=True` is used, Spark samples a portion of the data to guess column types. With messy or inconsistent date formats, this can cause several issues:

- **Wrong type inference:** If dates appear in mixed formats (e.g., `2024-01-15`, `15/01/2024`, `Jan 15 2024`), Spark may infer the column as `StringType` instead of `DateType` or `TimestampType`.
- **Data loss:** Some rows might parse correctly while others become `null` during later casts, causing data loss without any error.

**Best practice:** Always define the schema explicitly using `StructType` when the source data has complex or inconsistent formats.

---
### Q15: Final Processing Pipeline

In [ ]:
from pyspark.sql.functions import sum

final_result = (
    df
    .dropDuplicates()
    .na.fill({'price': 0})
    .groupBy('store_id')
    .agg(sum('price').alias('total_revenue'))
)

final_result.show()